# Optimización de hiperparámetros MDLF con Optuna

**Datos**: almacén `50mil_tarjetas_3_al_31_marzo`  
**Split**: temporal — train: 3-23 marzo / test: 24-31 marzo  
**Estrategia**: 40 trials × 5 épocas (proxy rápido), luego reentrenamiento con 20 épocas de la mejor config  
**Objetivo**: maximizar Top-1 accuracy en test

### Hiperparámetros en juego
| Grupo | Parámetros |
|---|---|
| LDA | K, alpha, beta, use_periods, min_df, log_counts, token_granularity, min_station_trips |
| CNN | n_heads, d_t, cnn_channels, kernel_size |
| Entrenamiento | optimizer, lr, weight_decay, lr_scheduler, n_weeks |
| Extensiones | include_prox, td_mode ('context'=paper / 'split'=alternativa), include_neighborhood_crowd |
| Normalización | norm (l2 / None) |

**LINE**: fijo con los mejores hiperparámetros del Optuna LINE (`line_cercanias_optuna.parquet`, 128 dims)

In [ ]:
import os, sys
from pathlib import Path

_cubo_dir = Path('/home/jupyter-lquijano/david/estudios/cubo_izquierdo').resolve()
_mdlf_dir = Path('/home/jupyter-lquijano/david/estudios/MDLF').resolve()

os.chdir(_cubo_dir)
for _p in [str(_cubo_dir), str(_mdlf_dir)]:
    if _p not in sys.path:
        sys.path.insert(0, _p)

print(f"CWD → {os.getcwd()}")

import optuna
import pandas as pd
import json
from datetime import datetime
from pipeline_mdlf import PipelineMDLF

optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# ── Configuración fija ────────────────────────────────────────────────────────

ALMACEN      = '50mil_tarjetas_3_al_31_marzo'
SPLIT_DATE   = '2025-03-24'   # train: 3–23 mar | test: 24–31 mar
N_EPOCHS_OPTUNA = 5
N_EPOCHS_FINAL  = 30
N_TRIALS        = 60

# Nombre fijo del estudio — siempre apunta al mismo .db al reanudar
STUDY_NAME = 'mdlf_marzo2025_v2'  # v2: td_mode añadido, norm=[l2/none], 60 trials
DB_PATH    = _mdlf_dir / 'runs' / f'{STUDY_NAME}.db'

# LINE: embeddings entrenados con la mejor config del Optuna LINE (128 dims)
LINE_AP_PATH = _mdlf_dir / 'LINE' / 'output' / 'line_cercanias_optuna.parquet'

# TTP CSVs para historial individual (enero + febrero + marzo)
# IndividualHistBuilder los procesa una sola vez y guarda caché en el almacén
_descomp = Path('/home/jupyter-lquijano/david/estudios/descomprimido')
TTP_PATHS = [
    str(_descomp / 'TTP_PERIODO_01_2025.csv'),   # enero
    str(_descomp / 'TTP_PERIODO_02_2025.csv'),   # febrero
    str(_descomp / 'TTP_PERIODO_03_2025.csv'),   # marzo
]

# Matrices OD para entrenar el LDA (región funcional estaciones)
OD_MATRICES_SUBDIRS = [
    'validaciones_3_7_febrero',
    'validaciones_8_9_febrero',
    'validaciones_10_16_febrero',
    'validaciones_17_23_febrero',
    'validaciones_24febrero_2marzo',
]

print(f"Almacén  : {ALMACEN}")
print(f"Split    : train < {SPLIT_DATE} | test >= {SPLIT_DATE}")
print(f"LINE     : {LINE_AP_PATH.name}  ({pd.read_parquet(LINE_AP_PATH).shape[1]} dims)")
print(f"Trials   : {N_TRIALS} × {N_EPOCHS_OPTUNA} épocas")
print(f"Study DB : {DB_PATH}")
print(f"TTP      : {len(TTP_PATHS)} períodos ({', '.join(Path(p).name for p in TTP_PATHS)})")

In [ ]:
# ── Función objetivo ─────────────────────────────────────────────────────────

def objective(trial: optuna.Trial) -> float:

    # ── LDA ──────────────────────────────────────────────────────────────────
    n_topics = trial.suggest_categorical('n_topics', [8, 16, 32, 48])

    lda_alpha_none = trial.suggest_categorical('lda_alpha_none', [True, False])
    lda_alpha = None if lda_alpha_none else trial.suggest_float('lda_alpha', 0.001, 1.0, log=True)

    lda_beta_none = trial.suggest_categorical('lda_beta_none', [True, False])
    lda_beta = None if lda_beta_none else trial.suggest_float('lda_beta', 0.001, 1.0, log=True)

    lda_use_periods       = trial.suggest_categorical('lda_use_periods', [False, True, 4])
    lda_min_df            = trial.suggest_int('lda_min_df', 1, 5)
    lda_log_counts        = trial.suggest_categorical('lda_log_counts', [True, False])
    lda_token_granularity = trial.suggest_categorical('lda_token_granularity', ['station', 'temporal'])
    lda_min_station_trips = trial.suggest_categorical('lda_min_station_trips', [0, 50, 100])

    # ── CNN ──────────────────────────────────────────────────────────────────
    n_heads      = trial.suggest_categorical('n_heads', [2, 4, 8])
    d_t          = trial.suggest_categorical('d_t', [8, 16, 32])
    cnn_channels = trial.suggest_categorical('cnn_channels', [8, 16, 32])
    kernel_size  = trial.suggest_categorical('kernel_size', [1, 3])

    # ── Entrenamiento ─────────────────────────────────────────────────────────
    optimizer    = trial.suggest_categorical('optimizer', ['sgd', 'adamw'])
    lr           = trial.suggest_float('lr', 1e-3, 1e-1, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-4, 1e-1, log=True)
    lr_scheduler = trial.suggest_categorical('lr_scheduler', [True, False])
    n_weeks      = trial.suggest_int('n_weeks', 1, 4)

    # ── Extensiones ───────────────────────────────────────────────────────────
    include_prox               = trial.suggest_categorical('include_prox', [True, False])
    td_mode                    = trial.suggest_categorical('td_mode', ['context', 'split'])
    include_neighborhood_crowd = trial.suggest_categorical('include_neighborhood_crowd', [True, False])

    # ── Normalización ─────────────────────────────────────────────────────────
    norm_str = trial.suggest_categorical('norm', ['l2', 'none'])
    norm = None if norm_str == 'none' else norm_str

    # ── Lanzar pipeline ───────────────────────────────────────────────────────
    pipe = PipelineMDLF(
        nombre_almacen=ALMACEN,
        ttp_paths=TTP_PATHS,
        od_matrices_subdirs=OD_MATRICES_SUBDIRS,
        line_ap_path=LINE_AP_PATH,
        split_date=SPLIT_DATE,
        n_topics_cercanias=n_topics,
        lda_alpha=lda_alpha,
        lda_beta=lda_beta,
        lda_use_periods=lda_use_periods,
        lda_min_df=lda_min_df,
        lda_log_counts=lda_log_counts,
        lda_token_granularity=lda_token_granularity,
        lda_min_station_trips=lda_min_station_trips,
        n_heads=n_heads,
        d_t=d_t,
        cnn_channels=cnn_channels,
        kernel_size=kernel_size,
        optimizer=optimizer,
        lr=lr,
        weight_decay=weight_decay,
        lr_scheduler=lr_scheduler,
        n_weeks=n_weeks,
        include_prox=include_prox,
        td_mode=td_mode,
        include_neighborhood_crowd=include_neighborhood_crowd,
        norm=norm,
        n_epochs=N_EPOCHS_OPTUNA,
        random_state=42,
    )

    pipe.run(run_name=f'optuna_t{trial.number:03d}')
    return pipe.test_accuracy

In [ ]:
# ── Ejecutar estudio Optuna ───────────────────────────────────────────────────
# load_if_exists=True → si el .db ya existe, reanuda desde donde se dejó

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42, n_startup_trials=25),
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=True,
)

trials_done = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f"Study    : {STUDY_NAME}")
print(f"DB       : {DB_PATH}")
print(f"Trials   : {trials_done} completados / {N_TRIALS} objetivo")
print()

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True,
)

print(f"\n✓ Estudio completado")
print(f"  Mejor Top-1 accuracy : {study.best_value:.4f}")
print(f"  Trial número         : {study.best_trial.number}")

In [ ]:
# ── Resultados ────────────────────────────────────────────────────────────────

print("=" * 60)
print("MEJORES HIPERPARÁMETROS")
print("=" * 60)
for k, v in study.best_trial.params.items():
    print(f"  {k:<40} {v}")
print(f"  {'Top-1 accuracy (5 épocas)':<40} {study.best_value:.4f}")

df_trials = study.trials_dataframe()
df_trials = df_trials.sort_values('value', ascending=False)
print(f"\nTop-10 trials:")
display(df_trials[['number', 'value', 'duration']].head(10))

In [ ]:
# ── Importancia de hiperparámetros ────────────────────────────────────────────
try:
    importances = optuna.importance.get_param_importances(study)
    print("Importancia de hiperparámetros (fANOVA):")
    for param, imp in importances.items():
        bar = '█' * int(imp * 50)
        print(f"  {param:<40} {imp:.4f}  {bar}")
except Exception as e:
    print(f"No se pudo calcular importancia: {e}")

In [ ]:
# ── Reentrenamiento con mejor configuración y épocas completas ────────────────

best = study.best_trial.params

_lda_alpha = None if best.get('lda_alpha_none', True) else best.get('lda_alpha')
_lda_beta  = None if best.get('lda_beta_none', True)  else best.get('lda_beta')
_norm      = None if best.get('norm', 'l2') == 'none' else best.get('norm', 'l2')

print(f"Reentrenando con la mejor configuración y {N_EPOCHS_FINAL} épocas...")
print()

pipe_best = PipelineMDLF(
    nombre_almacen=ALMACEN,
    ttp_paths=TTP_PATHS,
    od_matrices_subdirs=OD_MATRICES_SUBDIRS,
    line_ap_path=LINE_AP_PATH,
    split_date=SPLIT_DATE,
    n_topics_cercanias=best['n_topics'],
    lda_alpha=_lda_alpha,
    lda_beta=_lda_beta,
    lda_use_periods=best['lda_use_periods'],
    lda_min_df=best['lda_min_df'],
    lda_log_counts=best['lda_log_counts'],
    lda_token_granularity=best['lda_token_granularity'],
    lda_min_station_trips=best['lda_min_station_trips'],
    n_heads=best['n_heads'],
    d_t=best['d_t'],
    cnn_channels=best['cnn_channels'],
    kernel_size=best['kernel_size'],
    optimizer=best['optimizer'],
    lr=best['lr'],
    weight_decay=best['weight_decay'],
    lr_scheduler=best['lr_scheduler'],
    n_weeks=best['n_weeks'],
    include_prox=best['include_prox'],
    td_mode=best['td_mode'],
    include_neighborhood_crowd=best['include_neighborhood_crowd'],
    norm=_norm,
    n_epochs=N_EPOCHS_FINAL,
    random_state=42,
)

pipe_best.run(run_name='optuna_best_final')

print(f"\n{'='*50}")
print(f"RESULTADO FINAL (best config, {N_EPOCHS_FINAL} épocas)")
print(f"  Top-1 accuracy : {pipe_best.test_accuracy:.4f}")
print(f"  Top-5 accuracy : {pipe_best.test_accuracy5:.4f}")
print(f"  (Proxy 5 épocas era: {study.best_value:.4f})")

In [ ]:
# ── Guardar mejores params en JSON ────────────────────────────────────────────

_out = {
    'study_name': STUDY_NAME,
    'best_trial': study.best_trial.number,
    'top1_proxy_5epochs': study.best_value,
    'top1_final_20epochs': pipe_best.test_accuracy,
    'top5_final_20epochs': pipe_best.test_accuracy5,
    'best_params': {k: (None if v == 'none' else v) for k, v in best.items()},
}

_json_path = _mdlf_dir / 'runs' / f'{STUDY_NAME}_best.json'
with open(_json_path, 'w') as f:
    json.dump(_out, f, indent=2, default=str)

print(f"Resultados guardados en: {_json_path}")

In [ ]:
# ── Top combinaciones de hiperparámetros ──────────────────────────────────────

import pandas as pd

df_trials = study.trials_dataframe()
df_ok = df_trials[df_trials['state'] == 'COMPLETE'].copy()

param_cols = [c for c in df_ok.columns if c.startswith('params_')]
param_display = {c: c.replace('params_', '') for c in param_cols}

top_n = 15
df_top = (
    df_ok
    .sort_values('value', ascending=False)
    .head(top_n)
    [['number', 'value', 'duration'] + param_cols]
    .rename(columns={'number': 'trial', 'value': 'top1_proxy', **param_display})
    .reset_index(drop=True)
)

print(f'Top {top_n} trials (de {len(df_ok)} completados) — proxy 5 épocas')
print(f'Mejor Top-1 proxy : {df_ok["value"].max():.4f}')
print(f'Mediana           : {df_ok["value"].median():.4f}')
print()
pd.set_option('display.max_rows',10)
pd.set_option('display.max_columns',None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:.4f}' if abs(x) < 10 else f'{x:.2f}')
display(df_top)

# ── Tabla compacta — parámetros clave ────────────────────────────────────────
key_params = [
    'n_topics', 'lda_use_periods', 'lda_log_counts', 'lda_token_granularity',
    'n_heads', 'd_t', 'cnn_channels', 'kernel_size',
    'optimizer', 'lr', 'n_weeks',
    'include_prox', 'td_mode', 'include_neighborhood_crowd', 'norm',
]
available = ['trial', 'top1_proxy'] + [p for p in key_params if p in df_top.columns]

print()
print('── Parámetros clave ──────────────────────────────────────────────────────')
display(df_top[available])
